In [ ]:
import pandas as pd
import os
from zipfile import ZipFile
import requests

# ----------------------------
# 1. Download & Extract MovieLens 1M
# ----------------------------
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
dataset_zip = "ml-1m.zip"
extracted_path = "ml-1m"

if not os.path.exists(extracted_path):
    print("Downloading MovieLens 1M dataset...")
    r = requests.get(dataset_url)
    with open(dataset_zip, "wb") as f:
        f.write(r.content)
    with ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction done.")

# ----------------------------
# 2. Load all files
# ----------------------------
ratings_file = os.path.join(extracted_path, "ratings.dat")
movies_file = os.path.join(extracted_path, "movies.dat")
users_file  = os.path.join(extracted_path, "users.dat")

ratings = pd.read_csv(
    ratings_file, sep="::", engine="python",
    names=["UserID","MovieID","Rating","Timestamp"]
)
movies = pd.read_csv(
    movies_file, sep="::", engine="python",
    names=["MovieID","Title","Genres"], encoding="latin-1"
)
users = pd.read_csv(
    users_file, sep="::", engine="python",
    names=["UserID","Gender","Age","Occupation","Zip-code"]
)

# ----------------------------
# 3. Convert timestamp
# ----------------------------
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

# ----------------------------
# 4. Check for duplicates
# ----------------------------
print("Duplicate rows:")
print(f"Ratings: {ratings.duplicated().sum()}")
print(f"Movies:  {movies.duplicated().sum()}")
print(f"Users:   {users.duplicated().sum()}")

# Drop duplicates
ratings = ratings.drop_duplicates().reset_index(drop=True)
movies  = movies.drop_duplicates().reset_index(drop=True)
users   = users.drop_duplicates().reset_index(drop=True)

# ----------------------------
# 5. Check for missing values in detail
# ----------------------------
def report_missing(df, name):
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"No missing values in {name}.")
    else:
        print(f"Missing values in {name}:")
        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f" - Column '{col}': {missing_count} missing")
        print("Rows with missing values:")
        print(df[df.isnull().any(axis=1)])

report_missing(ratings, "ratings")
report_missing(movies, "movies")
report_missing(users, "users")

# ----------------------------
# 6. Filter ratings to May → Dec 2000
# ----------------------------
start_date = '2000-05-01'
end_date   = '2000-12-31'
ratings = ratings[(ratings['Timestamp'] >= start_date) & (ratings['Timestamp'] <= end_date)].reset_index(drop=True)
print("\nRatings after filtering May → Dec 2000:", ratings.shape)


In [ ]:
# ----------------------------
# Create monthly slices
# ----------------------------
monthly_bins = pd.date_range(start=start_date, end='2001-01-01', freq='MS')
ratings['SliceNum'] = pd.cut(ratings['Timestamp'], bins=monthly_bins, right=False).cat.codes + 1

# ----------------------------
# Inspect each slice
# ----------------------------
slice_summary = []

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    num_ratings = slice_df.shape[0]
    num_movies  = slice_df['MovieID'].nunique()
    num_users   = slice_df['UserID'].nunique()
    slice_summary.append({
        'SliceNum': s,
        'Ratings': num_ratings,
        'UniqueMovies': num_movies,
        'UniqueUsers': num_users
    })

slice_summary_df = pd.DataFrame(slice_summary)
print(slice_summary_df)


In [ ]:
import os
import pandas as pd

# ----------------------------
# 1. Create folder for slice metadata
# ----------------------------
os.makedirs("slice_metadata", exist_ok=True)

# ----------------------------
# 2. Split movies into popularity tiers per slice
# ----------------------------
high_pct, mid_pct, low_pct = 0.2, 0.3, 0.5  # 20/30/50 split

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    movie_counts = slice_df.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
    total_movies = len(movie_counts)
    
    # Determine cutoff indices
    high_cut  = int(total_movies * high_pct)
    mid_cut   = high_cut + int(total_movies * mid_pct)
    
    # Assign tiers
    movie_tiers = pd.DataFrame({'MovieID': movie_counts.index})
    movie_tiers['PopTier'] = 'low'  # default
    movie_tiers.loc[:high_cut-1, 'PopTier'] = 'high'
    movie_tiers.loc[high_cut:mid_cut-1, 'PopTier'] = 'medium'
    
    # Save slice metadata
    movie_tiers.to_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv", index=False)
    print(f"Slice {s}: {total_movies} movies → High:{high_cut}, Medium:{mid_cut-high_cut}, Low:{total_movies-mid_cut}")


In [ ]:
# ----------------------------
# User type assignment per slice
# ----------------------------
import os
import pandas as pd

slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata", exist_ok=True)

threshold = 0.7  # HighProportion threshold to classify mainstream

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty: 
        continue

    # Load movie popularity tiers for this slice
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    movie_tier_map = dict(zip(movie_tiers.MovieID, movie_tiers.PopTier))

    # Map each rating to movie tier
    slice_df['Tier'] = slice_df['MovieID'].map(movie_tier_map)

    # Count number of ratings per tier per user
    tier_counts = slice_df.groupby(['UserID', 'Tier']).size().unstack(fill_value=0)
    # Ensure all columns exist
    for col in ['high', 'medium', 'low']:
        if col not in tier_counts.columns:
            tier_counts[col] = 0

    # Compute HighProportion
    tier_counts['Total'] = tier_counts['high'] + tier_counts['medium'] + tier_counts['low']
    tier_counts['HighProportion'] = tier_counts['high'] / tier_counts['Total']

    # Assign user type
    tier_counts['UserType'] = tier_counts['HighProportion'].apply(
        lambda x: 'mainstream' if x >= threshold else 'niche'
    )

    # Save result
    user_type_df = tier_counts[['UserType']].reset_index()
    user_type_df.to_csv(f"slice_metadata/user_labels_slice_{s}.csv", index=False)
    
    print(f"Slice {s}: Total Users = {len(user_type_df)}, Mainstream = {sum(user_type_df.UserType=='mainstream')}, Niche = {sum(user_type_df.UserType=='niche')}")


In [ ]:
import pandas as pd
import os
import torch

# ----------------------------
# 1. KG construction per slice with relation types and mappings
# ----------------------------
slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata/kg", exist_ok=True)

# Define relation types
REL_USER_MOVIE = 0
REL_MOVIE_GENRE = 1
REL_MOVIE_POPTIER = 2
REL_USER_USERTYPE = 3

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty:
        continue
    
    print(f"\nConstructing KG for slice {s}...")

    # Load slice-specific metadata
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    user_types  = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv")
    
    # Extract unique nodes
    users = slice_df['UserID'].unique()
    movies_slice = slice_df['MovieID'].unique()
    genres = set()
    for g in movies['Genres']:
        genres.update(g.split('|'))
    genres = list(genres)
    
    pop_tiers = ['high', 'medium', 'low']
    utypes = ['mainstream', 'niche']
    
    # ----------------------------
    # Slice-local IDs
    user2nid = {u: i for i, u in enumerate(users)}
    movie2nid = {m: i + len(users) for i, m in enumerate(movies_slice)}
    genre2nid = {g: i + len(users) + len(movies_slice) for i, g in enumerate(genres)}
    pop2nid = {p: i + len(users) + len(movies_slice) + len(genres) for i, p in enumerate(pop_tiers)}
    utype2nid = {ut: i + len(users) + len(movies_slice) + len(genres) + len(pop_tiers) for i, ut in enumerate(utypes)}
    
    # Save mapping tables for hybrid warm-start later
    pd.DataFrame(list(user2nid.items()), columns=['UserID','NodeID']).to_csv(f"slice_metadata/kg/user_map_slice_{s}.csv", index=False)
    pd.DataFrame(list(movie2nid.items()), columns=['MovieID','NodeID']).to_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv", index=False)
    
    # ----------------------------
    # Build edges with relation types
    edges = []

    # User ↔ Movie
    for row in slice_df.itertuples():
        uid, mid = user2nid[row.UserID], movie2nid[row.MovieID]
        edges.append((uid, mid, REL_USER_MOVIE))
        edges.append((mid, uid, REL_USER_MOVIE))  # bi-directional

    # Movie ↔ Genre
    for row in movies.itertuples():
        if row.MovieID not in movie2nid:
            continue
        mid = movie2nid[row.MovieID]
        for g in row.Genres.split('|'):
            edges.append((mid, genre2nid[g], REL_MOVIE_GENRE))
            edges.append((genre2nid[g], mid, REL_MOVIE_GENRE))

    # Movie ↔ PopTier
    for row in movie_tiers.itertuples():
        mid = movie2nid[row.MovieID]
        edges.append((mid, pop2nid[row.PopTier], REL_MOVIE_POPTIER))
        edges.append((pop2nid[row.PopTier], mid, REL_MOVIE_POPTIER))

    # User ↔ UserType
    for row in user_types.itertuples():
        uid = user2nid[row.UserID]
        edges.append((uid, utype2nid[row.UserType], REL_USER_USERTYPE))
        edges.append((utype2nid[row.UserType], uid, REL_USER_USERTYPE))
    
    # Save edges with relation types
    edges_df = pd.DataFrame(edges, columns=['Src','Dst','RelType'])
    edges_df.to_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv", index=False)
    
    print(f"Slice {s} KG: {len(edges_df)} edges, {len(user2nid)} users, {len(movie2nid)} movies, {len(genre2nid)} genres, {len(pop2nid)} pop tiers, {len(utype2nid)} user types")


PROJECT README
Temporal Knowledge Graphs for Recommender Systems
================================================

1. What this project is about
-----------------------------

This project studies how user preferences and item popularity change over time,
and how recommender models can adapt to those changes instead of treating data as static.

Most recommender systems:
- Merge all interactions into one dataset
- Assume user behavior and item popularity are fixed
- Ignore temporal structure

That is unrealistic.

In this project, we:
- Split user–item interactions into time slices
- Build a separate Knowledge Graph (KG) for each slice
- Train models slice by slice
- Compare a baseline recommender with a more expressive final model

The goal is to evaluate whether modeling structure and time actually improves recommendations.


2. Why use a Knowledge Graph
----------------------------

A standard user–item matrix only captures:
“User A interacted with Movie B”.

A Knowledge Graph allows us to represent richer structure:
- Users have behavioral types (mainstream vs niche)
- Movies have genres
- Movies have popularity levels
- All of these can change over time

By encoding interactions as a graph:
- Simple graph models can be used as baselines
- Relation-aware models can exploit edge types
- The same data supports multiple model families


3. Why time slicing
-------------------

User behavior in May 2000 is not the same as in December 2000.

Instead of building one global graph, we:
- Split ratings into monthly time slices
- Build one KG per slice
- Allow users and movies to appear, disappear, or change context

This enables:
- Temporal evaluation
- Embedding warm-starting
- Studying preference drift


4. Models trained later
-----------------------

This preprocessing supports two categories of models:

1. Baseline model
   - Treats all edges as the same
   - Uses only graph structure
   - Ignores relation types

2. Final model
   - Uses relation types
   - Distinguishes different kinds of edges
   - Exploits richer KG structure

Both models use the same preprocessed data.


5. Dataset
----------

- MovieLens 1M
- Ratings filtered to May 2000 – December 2000
- Monthly time slices


6. Outputs per time slice
------------------------

For each slice, the following are created:
- Movie popularity tiers
- User behavioral types
- Slice-specific Knowledge Graph
- Node ID mappings for temporal alignment

All outputs are stored as CSV files.


7. Directory structure
----------------------

.
├── slice_metadata/
│   ├── movie_pop_tiers_slice_1.csv
│   ├── user_labels_slice_1.csv
│   ├── ...
│   └── kg/
│       ├── kg_edges_slice_1.csv
│       ├── user_map_slice_1.csv
│       ├── movie_map_slice_1.csv
│       └── ...


8. Movie popularity tiers
-------------------------

Files:
slice_metadata/movie_pop_tiers_slice_{s}.csv

Description:
- Movies are ranked by number of ratings within a slice
- Assigned to popularity tiers:
  - high
  - medium
  - low

These tiers capture short-term popularity, not global popularity.


9. User behavioral types
------------------------

Files:
slice_metadata/user_labels_slice_{s}.csv

Description:
- Users are labeled based on what they consume in that slice
- Users who mostly rate popular movies → mainstream
- Others → niche

User types can change across slices.


10. Knowledge Graphs
--------------------

Each slice has its own Knowledge Graph.


10.1 Edge list
--------------

Files:
slice_metadata/kg/kg_edges_slice_{s}.csv

Columns:
- Src     : source node ID
- Dst     : destination node ID
- RelType : relation type ID

All edges are stored as bidirectional.


10.2 Relation types
-------------------

Relation ID meanings:

0 → User ↔ Movie  
1 → Movie ↔ Genre  
2 → Movie ↔ Popularity Tier  
3 → User ↔ User Type  

Baseline models ignore RelType.
Final models use RelType.


11. Node IDs
------------

Node IDs are:
- Integers
- Unique within a slice
- Assigned in non-overlapping ranges by node type

This guarantees that users, movies, genres, tiers, and user types
cannot be confused with each other.


12. Node ID mappings (important)
--------------------------------

Files:
- slice_metadata/kg/user_map_slice_{s}.csv
- slice_metadata/kg/movie_map_slice_{s}.csv

These map original UserID / MovieID to slice-local node IDs.

They are required for:
- Matching entities across slices
- Warm-starting embeddings
- Temporal stabilization during training


13. What is NOT stored
----------------------

- No trained models
- No tensors
- No pickled Python objects
- No framework-specific formats

Everything is reconstructed from CSVs in training notebooks.


14. How a new training session should start
-------------------------------------------

A fresh session should:
1. Select a slice
2. Load KG edges and mappings
3. Convert data to tensors as needed
4. Train baseline and final models
5. Optionally align embeddings across slices


15. Summary
-----------

This project builds time-aware Knowledge Graphs from MovieLens data
to compare a baseline recommender against a relation-aware model
while explicitly modeling temporal dynamics.
